# TRACK-FA Progression-First Deep Learning Pipeline

This notebook trains neural progression models with imaging and demographic inputs only. Clinical scores are retained only for evaluation summaries; they are not used as model features, training targets, or auxiliary-head losses.


In [1]:
# Seed numpy, random, and torch before creating any stochastic split or model.
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "src").exists() and (path / "data").exists():
            return path
    raise FileNotFoundError("Could not find the project root containing src/ and data/.")


REPO_ROOT = find_project_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import Config, DEFAULT_CONFIG, set_global_seeds

set_global_seeds(42)


In [ ]:
# Import analysis, modeling, and plotting tools; then configure training settings.
import time
import warnings

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler

from src.data.qc import filter_complete_pairs
from src.data.trackfa_pairs import infer_trackfa_feature_groups, trackfa_pairs_to_long
from src.eval.metrics import paired_cohens_d, srm, paired_ttest, rmse
from src.eval.shap import _pick_best_combo, run_shap_on_combo, TOP_N_FEATURES
from src.training.fusion import (
    evaluate_fusion_loss,
    prepare_fusion_arrays,
    train_fusion_model,
)
from src.training.pair import prepare_pair_arrays, train_pair_model
from src.viz.plots import _plot_modality_summary, _plot_top_pos_neg

warnings.filterwarnings('ignore')

config = DEFAULT_CONFIG
DEVICE = torch.device('cpu')
RANDOM_SEED = config.random_state

# Compact training settings keep iteration fast; increase epochs for full runs.
EPOCHS = 20
PATIENCE = 4
LR = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.2
VAL_FRACTION = 0.2
USE_CLINICAL_HEADS = False
LAMBDA_FARS = 0.0
LAMBDA_SARA = 0.0
LAMBDA_PROG = 1.0

TRAIN_KWARGS = dict(
    epochs=EPOCHS,
    patience=PATIENCE,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    dropout=DROPOUT,
    val_fraction=VAL_FRACTION,
    seed=RANDOM_SEED,
    use_clinical_heads=USE_CLINICAL_HEADS,
    lambda_prog=LAMBDA_PROG,
    lambda_fars=LAMBDA_FARS,
    lambda_sara=LAMBDA_SARA,
)

DATA_PATH = config.processed_data_dir / 'trackfa_pairs_drop3poms.csv'
print('Working directory:', config.repo_root)
print('Data path:', DATA_PATH, 'OK' if DATA_PATH.exists() else 'MISSING')


In [3]:
# Load the paired TRACK-FA table and convert it to visit-level rows.
pairs_df = pd.read_csv(DATA_PATH)
groups = infer_trackfa_feature_groups(pairs_df)
long_df = trackfa_pairs_to_long(pairs_df)
subject_col = 'pair_id'

background = list(groups.background)
structural = list(groups.poms + groups.brainspinemorph)
structural_ext = list(structural)
diffusion = list(groups.braindti)

print('Pairs:', pairs_df.shape)
print('Long rows:', len(long_df), 'pairs:', long_df[subject_col].nunique())
print('Feature counts:', {
    'background': len(background),
    'structural': len(structural),
    'diffusion': len(diffusion),
})


Pairs: (207, 455)
Long rows: 414 pairs: 207
Feature counts: {'background': 6, 'structural': 42, 'diffusion': 108}


In [ ]:
# Define the modality combinations to evaluate.
all_combinations = [
    {'name': 'background', 'features': background},
    {'name': 'structural', 'features': structural},
    {'name': 'diffusion', 'features': diffusion},
    {'name': 'background_diffusion', 'features': background + diffusion},
    {'name': 'background_structural', 'features': background + structural},
    {'name': 'background_structural_ext', 'features': background + structural_ext},
    {'name': 'background_structural_diffusion', 'features': background + structural + diffusion},
    {'name': 'background_structural_ext_diffusion', 'features': background + structural_ext + diffusion},
]
combinations = [c for c in all_combinations if len(c['features']) > 0]

combo_meta = {}
for combo in combinations:
    feats = combo['features']
    combo_meta[combo['name']] = {
        'features': feats,
        'struct_idx': [i for i, f in enumerate(feats) if f in structural_ext],
        'diff_idx': [i for i, f in enumerate(feats) if f in diffusion],
        'back_idx': [i for i, f in enumerate(feats) if f in background],
    }

print('Combinations:', [c['name'] for c in combinations])


Combinations: ['background', 'structural', 'diffusion', 'background_diffusion', 'background_structural', 'background_structural_ext', 'background_structural_diffusion', 'background_structural_ext_diffusion']


: 

In [ ]:
# Train FusionMLP for each modality combination with subject-level leave-one-out evaluation.
fusion_results = []

for combo in combinations:
    feats = combo['features']
    meta = combo_meta[combo['name']]
    sub = filter_complete_pairs(long_df, subject_col, feats)
    subjects = sub[subject_col].unique()

    oof_rows = []
    fars_true, fars_pred = [], []
    sara_true, sara_pred = [], []
    epochs_used = []
    start_time = time.time()

    for sid in subjects:
        train = sub[sub[subject_col] != sid]
        test = sub[sub[subject_col] == sid]

        scaler = StandardScaler()
        scaler.fit(train[feats].values)

        model, best_epoch = train_fusion_model(
            train, feats, meta, scaler,
            subject_col=subject_col, device=DEVICE,
            **TRAIN_KWARGS,
        )
        epochs_used.append(best_epoch)
        model.eval()

        test_arrays = prepare_fusion_arrays(
            test, feats, meta, scaler,
            subject_col=subject_col, device=DEVICE,
        )
        with torch.inference_mode():
            _, fars_out, sara_out, prog_out = evaluate_fusion_loss(
                model, test_arrays,
                use_clinical_heads=USE_CLINICAL_HEADS,
                lambda_prog=LAMBDA_PROG,
                lambda_fars=LAMBDA_FARS,
                lambda_sara=LAMBDA_SARA,
            )

        prog_np = prog_out.detach().cpu().numpy().ravel()
        fars_np = fars_out.detach().cpu().numpy().ravel()
        sara_np = sara_out.detach().cpu().numpy().ravel()

        for v, sc in zip(test['visit'].values, prog_np):
            oof_rows.append({'subject': sid, 'visit': v, 'score': sc, 'method': 'FusionMLP', 'combination': combo['name']})

        fars_true.extend(test['FARS'].values)
        fars_pred.extend(fars_np)
        sara_true.extend(test['SARA'].values)
        sara_pred.extend(sara_np)

    runtime = time.time() - start_time
    oof_df = pd.DataFrame(oof_rows)
    d, mean_diff, sd_diff, n = paired_cohens_d(oof_df, 'subject')
    srm_val = srm(oof_df, 'subject')
    p = paired_ttest(oof_df, 'subject')

    fars_r2 = np.nan; fars_rmse_v = np.nan; sara_r2 = np.nan; sara_rmse_v = np.nan
    if USE_CLINICAL_HEADS:
        yft = np.asarray(fars_true, dtype=float)
        yfp = np.asarray(fars_pred, dtype=float)
        m = np.isfinite(yft) & np.isfinite(yfp)
        if m.sum() >= 3:
            fars_r2 = r2_score(yft[m], yfp[m])
            fars_rmse_v = rmse(yft[m], yfp[m])
        yst = np.asarray(sara_true, dtype=float)
        ysp = np.asarray(sara_pred, dtype=float)
        m = np.isfinite(yst) & np.isfinite(ysp)
        if m.sum() >= 3:
            sara_r2 = r2_score(yst[m], ysp[m])
            sara_rmse_v = rmse(yst[m], ysp[m])

    fusion_results.append({
        'model': 'FusionMLP',
        'combination': combo['name'],
        'd': d, 'srm': srm_val,
        'mean_diff': mean_diff, 'sd_diff': sd_diff, 'p_value': p,
        'fars_r2': fars_r2, 'fars_rmse': fars_rmse_v,
        'sara_r2': sara_r2, 'sara_rmse': sara_rmse_v,
        'runtime_sec': runtime,
        'mean_epochs': float(np.mean(epochs_used)),
    })

fusion_results_df = pd.DataFrame(fusion_results)
fusion_results_df

In [ ]:
# Train PairModel on baseline/follow-up pairs for each modality combination.
pair_results = []

for combo in combinations:
    feats = combo['features']
    sub = filter_complete_pairs(long_df, subject_col, feats)
    subjects = sub[subject_col].unique()

    oof_rows = []
    epochs_used = []
    fars_true, fars_pred = [], []
    sara_true, sara_pred = [], []
    start_time = time.time()

    for sid in subjects:
        train = sub[sub[subject_col] != sid]
        test = sub[sub[subject_col] == sid]

        scaler = StandardScaler()
        scaler.fit(train[feats].values)

        model, best_epoch = train_pair_model(
            train, feats, scaler,
            subject_col=subject_col, device=DEVICE,
            **TRAIN_KWARGS,
        )
        epochs_used.append(best_epoch)
        model.eval()

        g = test.sort_values('visit')
        x1 = torch.tensor(scaler.transform(g[g['visit'] == 1][feats].values), dtype=torch.float32, device=DEVICE)
        x2 = torch.tensor(scaler.transform(g[g['visit'] == 2][feats].values), dtype=torch.float32, device=DEVICE)

        with torch.inference_mode():
            prog, fars1, fars2, sara1, sara2 = model(x1, x2)

        score = prog.detach().cpu().numpy().ravel()[0]
        if USE_CLINICAL_HEADS:
            fars_true.extend(g['FARS'].values)
            sara_true.extend(g['SARA'].values)
            fars_pred.extend([float(fars1.detach().cpu().numpy().ravel()[0]), float(fars2.detach().cpu().numpy().ravel()[0])])
            sara_pred.extend([float(sara1.detach().cpu().numpy().ravel()[0]), float(sara2.detach().cpu().numpy().ravel()[0])])
        oof_rows.append({'subject': sid, 'visit': 1, 'score': -0.5 * score, 'method': 'PairModel', 'combination': combo['name']})
        oof_rows.append({'subject': sid, 'visit': 2, 'score': 0.5 * score, 'method': 'PairModel', 'combination': combo['name']})

    runtime = time.time() - start_time
    oof_df = pd.DataFrame(oof_rows)
    d, mean_diff, sd_diff, n = paired_cohens_d(oof_df, 'subject')
    srm_val = srm(oof_df, 'subject')
    p = paired_ttest(oof_df, 'subject')

    fars_r2 = np.nan; fars_rmse_v = np.nan; sara_r2 = np.nan; sara_rmse_v = np.nan
    if USE_CLINICAL_HEADS:
        yft = np.asarray(fars_true, dtype=float)
        yfp = np.asarray(fars_pred, dtype=float)
        m = np.isfinite(yft) & np.isfinite(yfp)
        if m.sum() >= 3:
            fars_r2 = r2_score(yft[m], yfp[m])
            fars_rmse_v = rmse(yft[m], yfp[m])
        yst = np.asarray(sara_true, dtype=float)
        ysp = np.asarray(sara_pred, dtype=float)
        m = np.isfinite(yst) & np.isfinite(ysp)
        if m.sum() >= 3:
            sara_r2 = r2_score(yst[m], ysp[m])
            sara_rmse_v = rmse(yst[m], ysp[m])

    pair_results.append({
        'model': 'PairModel',
        'combination': combo['name'],
        'd': d, 'srm': srm_val,
        'mean_diff': mean_diff, 'sd_diff': sd_diff, 'p_value': p,
        'fars_r2': fars_r2, 'fars_rmse': fars_rmse_v,
        'sara_r2': sara_r2, 'sara_rmse': sara_rmse_v,
        'runtime_sec': runtime,
        'mean_epochs': float(np.mean(epochs_used)),
    })

pair_results_df = pd.DataFrame(pair_results)
pair_results_df

In [ ]:
# Aggregate paired Cohen's d results and display the model comparison table.
summary_df = (
    pd.concat([fusion_results_df, pair_results_df], ignore_index=True)
      .sort_values(['model', 'combination'])
      .reset_index(drop=True)
)

summary_df


In [ ]:
# Select the strongest broad-modality combinations for SHAP interpretation.
best_fusion_combo, fusion_full = _pick_best_combo(fusion_results_df, combo_meta, prefer_all_modalities=True)
best_pair_combo, pair_full = _pick_best_combo(pair_results_df, combo_meta, prefer_all_modalities=True)
print('Best fusion combo:', best_fusion_combo, '(all modalities)' if fusion_full else '(fallback)')
print('Best pair combo:', best_pair_combo, '(all modalities)' if pair_full else '(fallback)')

In [ ]:
# Explain the FusionMLP progression head with SHAP values.
fusion_shap = run_shap_on_combo(
    model_kind='fusion',
    combo_name=best_fusion_combo,
    combinations=combinations,
    combo_meta=combo_meta,
    long_df=long_df,
    subject_col=subject_col,
    device=DEVICE,
    seed=RANDOM_SEED,
    train_kwargs=TRAIN_KWARGS,
)

_plot_modality_summary(
    fusion_shap['modality_summary'],
    title=f"FusionMLP modality summary (sum(mean(SHAP))) (combo={best_fusion_combo})",
)
_plot_top_pos_neg(
    fusion_shap['back_mean'], fusion_shap['back_names'],
    f'FusionMLP BACKGROUND (combo={best_fusion_combo})',
)
_plot_top_pos_neg(
    fusion_shap['struct_mean'], fusion_shap['struct_names'],
    f'FusionMLP STRUCTURAL (combo={best_fusion_combo})',
)
_plot_top_pos_neg(
    fusion_shap['diff_mean'], fusion_shap['diff_names'],
    f'FusionMLP DIFFUSION (combo={best_fusion_combo})',
)


In [ ]:
# Explain the PairModel progression head with SHAP values.
pair_shap = run_shap_on_combo(
    model_kind='pair',
    combo_name=best_pair_combo,
    combinations=combinations,
    combo_meta=combo_meta,
    long_df=long_df,
    subject_col=subject_col,
    device=DEVICE,
    seed=RANDOM_SEED,
    train_kwargs=TRAIN_KWARGS,
)

_plot_modality_summary(
    pair_shap['modality_summary'],
    title=f"PairModel modality summary (sum(mean(SHAP))) (combo={best_pair_combo})",
)
if pair_shap['back_mean'].size:
    _plot_top_pos_neg(
        pair_shap['back_mean'], pair_shap['back_names'],
        f'PairModel BACKGROUND (combo={best_pair_combo})',
    )
else:
    print('[skip] PairModel BACKGROUND (no features)')
if pair_shap['struct_mean'].size:
    _plot_top_pos_neg(
        pair_shap['struct_mean'], pair_shap['struct_names'],
        f'PairModel STRUCTURAL (combo={best_pair_combo})',
    )
else:
    print('[skip] PairModel STRUCTURAL (no features)')
if pair_shap['diff_mean'].size:
    _plot_top_pos_neg(
        pair_shap['diff_mean'], pair_shap['diff_names'],
        f'PairModel DIFFUSION (combo={best_pair_combo})',
    )
else:
    print('[skip] PairModel DIFFUSION (no features)')


## Final summary

Display the model comparison table and SHAP interpretation plots directly in the notebook.


In [ ]:
print('Summary rows:', len(summary_df))
print('Best FusionMLP combo:', best_fusion_combo)
print('Best PairModel combo:', best_pair_combo)
summary_df.sort_values('d', ascending=False)
